# BAW on BODMAS — reproducible Colab entry point

This notebook is intentionally thin. The implementation lives in `src/baw/`, so the GitHub code and the Colab run use the same source files.

**Scope:** feature-space research prototype; optimized trigger vectors are not claimed to be valid PE files.


## 1. Clone and install

Replace the placeholder URL after creating the GitHub repository.


In [ ]:
REPO_URL = "https://github.com/<YOUR_USERNAME>/baw-malware-watermark.git"

!git clone -q "$REPO_URL"
%cd baw-malware-watermark
!python -m pip install -q -e .


## 2. Imports and configuration


In [ ]:
from baw.config import Config
from baw.main_real import (
    aggregate_track_a,
    get_bodmas,
    print_track_a_summary,
    run_track_a,
    run_track_b_ablations,
    run_track_b_stealth,
    run_track_b_surrogate,
)
from baw.figures_real import (
    fig_ablations,
    fig_stealth,
    fig_surrogate,
    fig_track_a_comparison,
)

cfg = Config()
QUICK_TEST = False

if QUICK_TEST:
    cfg.n_seeds = 2
    cfg.track_a_owner_n = 2_000
    cfg.track_a_reference_n = 1_500
    cfg.track_a_test_n = 1_000
    cfg.track_b_owner_n = 3_000
    cfg.track_b_reference_n = 2_000
    cfg.track_b_test_n = 1_500
    cfg.ablation_K = (50, 150)
    cfg.ablation_eps = (0.15, 0.30)
    cfg.ablation_wm_weight = (1.0, 2.0)
    cfg.n_epochs_base = 8
    cfg.wm_epochs = 5
    cfg.ft_epochs = 5
    cfg.distill_epochs = 6
    cfg.fine_prune_ft_epochs = 4

print(cfg)


## 3. Load BODMAS


In [ ]:
X, y, timestamps = get_bodmas(cfg)
print("Shape:", X.shape)
print("Benign:", int((y == 0).sum()), "Malware:", int((y == 1).sum()))


## 4. Track A — multi-seed core comparison


In [ ]:
per_seed_results = run_track_a(X, y, timestamps, cfg)
track_a_summary, track_a_comparisons = aggregate_track_a(per_seed_results, cfg)
print_track_a_summary(track_a_summary, track_a_comparisons, cfg)
fig_track_a_comparison(track_a_summary, track_a_comparisons)


## 5. Track B — diagnostic experiments


In [ ]:
ablations, track_b_data = run_track_b_ablations(X, y, timestamps, cfg)
fig_ablations(ablations)

surrogate_result = run_track_b_surrogate(cfg, track_b_data)
fig_surrogate(surrogate_result)

stealth_result = run_track_b_stealth(cfg, track_b_data)
fig_stealth(stealth_result)


## 6. Save exact results


In [ ]:
import json
import os
import platform
import subprocess
import sys

os.makedirs(cfg.outdir, exist_ok=True)

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip()
except Exception:
    git_commit = None

out = {
    "metadata": {
        "git_commit": git_commit,
        "python": sys.version,
        "platform": platform.platform(),
    },
    "config": dict(cfg.__dict__),
    "track_a_per_seed": per_seed_results,
    "track_a_summary": track_a_summary,
    "track_a_comparisons": track_a_comparisons,
    "track_b_ablations": ablations,
    "track_b_surrogate": surrogate_result,
    "track_b_stealth": stealth_result,
}

path = os.path.join(cfg.outdir, "results_real.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, default=str)

print("Saved:", path)
